# Lab Buổi 06: Interface Programming & API Integration
> **Đối tượng:** Team AI ProPTIT D24  
> **Tài liệu tham khảo:** [lecture_notes.md](../lecture_notes.md) | [README_Lab.md](README_Lab.md)

---

### 🛠️ Hướng dẫn Chuẩn bị Môi trường
Chạy cell bên dưới để cài đặt các thư viện cần thiết (`fastapi`, `uvicorn`, `requests`, `streamlit`, `gradio`, `pydantic`, `httpx`).

In [ ]:
# !pip install fastapi uvicorn requests streamlit gradio pydantic httpx


## Task 1: Demo Web UI nhanh với Gradio (Beginner - 20%)
- **Mục tiêu:** Viết hàm giả lập `mock_predict(text)` và bọc nhanh thành giao diện Web bằng `gr.Interface`.
- **Tham khảo:** Xem mục 2.1 & 3.1 trong `lecture_notes.md`.

In [ ]:
import gradio as gr

def mock_predict(text: str) -> str:
    """
    Hàm giả lập dự đoán cảm xúc văn bản.
    Input: text (str)
    Output: chuỗi kết quả có dạng 'AI dự đoán: [TÍCH CỰC/TIÊU CỰC]'
    """
    ### TODO: Viết code của bạn tại đây
    # 1. Kiểm tra nếu text rỗng (sau khi strip) -> trả về 'Vui lòng nhập văn bản hợp lệ!'
    # 2. Xử lý logic giả lập: Nếu độ dài text (len) là số chẵn -> 'TÍCH CỰC', ngược lại -> 'TIÊU CỰC'
    # 3. Trả về chuỗi kết quả theo đúng định dạng
    pass

# Tạo Gradio Interface
### TODO: Tạo đối tượng gr.Interface bọc hàm mock_predict với inputs='text' và outputs='text'
demo = None  # Thay thế None bằng gr.Interface(...)

# Chạy thử UI trên Colab/Local
# if demo: demo.launch()


## Task 2: Xây dựng Backend API với FastAPI & Pydantic Validation (Intermediate - 25%)
- **Mục tiêu:** Định nghĩa Schema dữ liệu với Pydantic và tạo Endpoint POST `/predict` trả về kết quả JSON.
- **Tham khảo:** Xem mục 3.1 trong `lecture_notes.md`.
- **Lưu ý:** Chúng ta sử dụng `TestClient` của FastAPI để kiểm thử API trực tiếp trong cell mà không cần chạy server uvicorn.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from fastapi.testclient import TestClient

app = FastAPI(title="AI Model Serving API")

# 1. Định nghĩa Pydantic Schema cho Request
class TextRequest(BaseModel):
    ### TODO: Định nghĩa trường `text` thuộc kiểu str, bắt buộc nhập (Field min_length=1)
    pass

# 2. Định nghĩa Pydantic Schema cho Response
class TextResponse(BaseModel):
    status: str
    predicted_label: str
    confidence: float

# 3. Tạo Endpoint POST /predict
@app.post("/predict", response_model=TextResponse)
def predict(data: TextRequest):
    """
    Endpoint nhận JSON payload, kiểm tra dữ liệu và trả về kết quả dự đoán.
    """
    ### TODO: Viết code xử lý tại đây:
    # - Nếu data.text rỗng -> raise HTTPException(status_code=400, detail='Text rỗng')
    # - Logic giả lập: label = 'POSITIVE' nếu len(raw_text) % 2 == 0 ngược lại 'NEGATIVE'
    # - Trả về đối tượng TextResponse(status='success', predicted_label=label, confidence=0.95)
    pass

# Kiểm thử API bằng TestClient
# client = TestClient(app)
# response = client.post("/predict", json={"text": "Hello ProPTIT"})
# print("Status Code:", response.status_code)
# print("Response JSON:", response.json())


## Task 3: Streaming API Endpoint với FastAPI (Intermediate - 25%)
- **Mục tiêu:** Viết Async Generator nhả từng token và endpoint `/predict-stream` trả về `StreamingResponse`.
- **Tham khảo:** Xem mục 2.4 & 3.3 trong `lecture_notes.md`.

In [ ]:
import asyncio
from typing import AsyncGenerator
from fastapi.responses import StreamingResponse

async def token_generator(prompt: str) -> AsyncGenerator[str, None]:
    """
    Generator bất đồng bộ nhả ra từng token từ chuỗi prompt.
    """
    tokens = prompt.split()
    ### TODO: Lặp qua từng token trong `tokens`:
    # - yield từng token kèm khoảng trắng phía sau: f"{token} "
    # - await asyncio.sleep(0.1) để mô phỏng độ trễ sinh từ của LLM
    pass

@app.get("/predict-stream")
async def predict_stream(text: str):
    """
    Endpoint trả về dữ liệu luồng StreamingResponse.
    """
    ### TODO: Trả về StreamingResponse bọc lấy token_generator(text) với media_type='text/plain'
    pass

# Kiểm thử Streaming API bằng TestClient
# with client.stream("GET", "/predict-stream?text=Deep Learning PyTorch") as response:
#     for chunk in response.iter_text():
#         print(chunk, end="")


## Task 4: Streamlit Client UI kết nối Backend & Quản lý Session State (Advanced - 30%)
- **Vấn đề:** Streamlit và Uvicorn (FastAPI) là các ứng dụng web cần chạy bằng command line riêng biệt, không thể chạy trực tiếp trong một cell Notebook.
- **Giải pháp:** Sử dụng magic command `%%writefile` để xuất code ra các file `.py`, sau đó chạy qua Terminal.

### 4.1. Xuất code FastAPI Backend
Hãy copy toàn bộ code định nghĩa FastAPI app từ Task 2 và Task 3 dán vào cell dưới đây để tạo file `backend_app.py`.

In [ ]:
%%writefile backend_app.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import asyncio
from typing import AsyncGenerator
from fastapi.responses import StreamingResponse

app = FastAPI(title="AI Model Serving API")

### TODO: Paste các class Pydantic và 2 endpoint (/predict, /predict-stream) của bạn vào đây


### 4.2. Viết code Streamlit Frontend
Hoàn thiện code Streamlit kết nối tới API vừa tạo.

In [ ]:
%%writefile frontend_app.py
import requests
import streamlit as st

API_URL = "http://localhost:8000"
st.set_page_config(page_title="AI Client", page_icon="🚀")

# 1. Khởi tạo Session State
### TODO: Kiểm tra nếu 'history' chưa có trong st.session_state thì khởi tạo st.session_state.history = []

st.title("AI Model Client - Streamlit")
tab1, tab2 = st.tabs(["JSON API (Standard)", "Streaming API (Real-time)"])

with tab1:
    user_input = st.text_input("Nhập dữ liệu:")
    if st.button("Gửi POST Request"):
        ### TODO: Gửi requests.post tới f'{API_URL}/predict' với json={'text': user_input}
        # - Nếu response.status_code == 200: lấy data = response.json(), hiển thị st.success,
        #   và lưu dict {'input': user_input, 'output': data['predicted_label']} vào st.session_state.history
        # - Ngược lại: hiển thị st.error
        pass

with tab2:
    stream_input = st.text_input("Nhập prompt stream:")
    if st.button("Bắt đầu Stream"):
        ### TODO: Định nghĩa generator consumer để đọc luồng byte từ requests.get(..., stream=True)
        # def stream_consumer():
        #     with requests.get(f'{API_URL}/predict-stream', params={'text': stream_input}, stream=True) as r:
        #         for chunk in r.iter_content(chunk_size=None, decode_unicode=True):
        #             if chunk: yield chunk
        # st.write_stream(stream_consumer)
        pass

### TODO: Nếu st.session_state.history có dữ liệu, hiển thị danh sách lịch sử bằng st.dataframe()


### 4.3. Chạy Ứng dụng
Mở 2 cửa sổ Terminal (hoặc Command Prompt) trong cùng thư mục chứa notebook này và chạy:

**Terminal 1 (Backend):**
```bash
uvicorn backend_app:app --reload --port 8000
```

**Terminal 2 (Frontend):**
```bash
streamlit run frontend_app.py
```

Truy cập vào URL localhost do Streamlit cung cấp (thường là `http://localhost:8501`) để test kết quả!